In [1]:
from google.colab import drive
import os
import re
import csv
from pathlib import Path

drive.mount('/content/drive')

caminho_cnv = '/content/drive/MyDrive/datasus/cbo/CBO2002.CNV'

pasta_saida = '/content/drive/MyDrive/datasus/cbo/'

Mounted at /content/drive


In [2]:
def ler_cnv(caminho):

    nome = os.path.basename(caminho).upper()

    with open(caminho, "r", encoding="cp1252") as arquivo:
        linhas = arquivo.read().splitlines()

    if not linhas:
        raise ValueError("Arquivo CNV vazio.")

    m = re.match(r"^\s*(\d+)", linhas[0])

    if not m:
        raise ValueError(f"Cabeçalho inválido em {nome}")

    quantidade_declarada = int(m.group(1))

    registros = []

    for numero_linha, linha in enumerate(linhas[1:], start=2):

        if not linha.strip():
            continue

        # Extrair ordem
        m = re.match(r"^\s*(\d+)\s{2}", linha)

        if not m or len(linha) < 60:
            raise ValueError(
                f"{nome}, linha {numero_linha}: registro inválido"
            )

        ordem = m.group(1)

        # Extrair descrição
        descricao = linha[m.end():60].strip()

        # Extrair código
        codigo = linha[60:].strip()

        if not descricao or not codigo:
            raise ValueError(
                f"{nome}, linha {numero_linha}: descrição/código vazio"
            )

        registros.append([ordem, descricao, codigo])

    # Conferir quantidade de registros
    if len(registros) != quantidade_declarada:
        raise ValueError(
            f"{nome}: declarados {quantidade_declarada}, "
            f"lidos {len(registros)}"
        )

    return registros


# REALIZAR TRANSCRIÇÃO
registros = ler_cnv(caminho_cnv)

# DEFINIR NOME DO CSV
nome_base = Path(caminho_cnv).stem

caminho_csv = os.path.join(
    pasta_saida,
    nome_base + ".csv"
)

# SALVAR CSV
with open(
    caminho_csv,
    "w",
    encoding="utf-8-sig",
    newline=""
) as arquivo:

    escritor = csv.writer(arquivo, delimiter=";")

    escritor.writerow(["ordem", "descricao", "codigo"])
    escritor.writerows(registros)


# RESULTADO
print("TRANSCRIÇÃO CONCLUÍDA!")
print(f"Arquivo: {caminho_csv}")
print(f"Registros transcritos: {len(registros)}")

TRANSCRIÇÃO CONCLUÍDA!
Arquivo: /content/drive/MyDrive/datasus/cbo/CBO2002.csv
Registros transcritos: 2429
